In [ ]:
####################################
#ENVIRONMENT SETUP

In [ ]:
#Importing Libraries
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
import matplotlib.cm as cm
from matplotlib.colors import Normalize
from matplotlib.ticker import MaxNLocator
from matplotlib.ticker import ScalarFormatter
import matplotlib.gridspec as gridspec
from matplotlib.lines import Line2D
import xarray as xr

import sys; import os; import time; from datetime import timedelta
import pickle
import h5py
from tqdm import tqdm

In [ ]:
#MAIN DIRECTORIES
def GetDirectories():
    mainDirectory='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
    mainCodeDirectory=os.path.join(mainDirectory,"Code/CodeFiles/")
    scratchDirectory='/mnt/lustre/koa/scratch/air673/'
    codeDirectory=os.getcwd()
    return mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory

[mainDirectory,mainCodeDirectory,scratchDirectory,codeDirectory] = GetDirectories()

In [ ]:
def GetPlottingDirectory(plotFileName, plotType, folderName):
    plottingDirectory = mainCodeDirectory=os.path.join(mainDirectory,"Code","PLOTTING")
    
    specificPlottingDirectory = os.path.join(plottingDirectory, plotType, 
                                             f"{ModelData.res}_{ModelData.t_res}_{ModelData.Nz_str}nz",folderName)
    os.makedirs(specificPlottingDirectory, exist_ok=True)

    plottingFileName=os.path.join(specificPlottingDirectory, plotFileName)

    return plottingFileName

def SaveFigure(fig,plotType, folderName,fileName):
    plotFileName = f"{fileName}_{ModelData.res}_{ModelData.t_res}_{ModelData.Np_str}.jpg"
    plottingFileName = GetPlottingDirectory(plotFileName, plotType, folderName)
    print(f"Saving figure to {plottingFileName}")
    fig.savefig(plottingFileName, dpi=300, bbox_inches='tight')
    plt.close(fig)

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
from CLASSES_Variable_Calculation import ModelData_Class, SlurmJobArray_Class, DataManager_Class

In [ ]:
#IMPORT FUNCTIONS
sys.path.append(os.path.join(mainCodeDirectory,"2_Variable_Calculation"))
import FUNCTIONS_Variable_Calculation
from FUNCTIONS_Variable_Calculation import *

In [ ]:
#data loading class
ModelData = ModelData_Class(mainDirectory, scratchDirectory, simulationNumber=3)
#data manager class
DataManager = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracking_Algorithms", dataName="Lagrangian_UpdraftTracking",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#data manager class (for saving data)
DataManager_TrackedProfiles = DataManager_Class(mainDirectory, scratchDirectory, ModelData.res, ModelData.t_res, ModelData.Nz_str,
                                ModelData.Np_str, dataType="Tracked_Profiles", dataName="Tracked_Profiles",
                                dtype='float32',codeSection = "Project_Algorithms")

In [ ]:
#IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","2_Tracking_Algorithms"))
from CLASSES_TrackingAlgorithms import TrackingAlgorithms_DataLoading_Class, Results_InputOutput_Class, TrackedParcel_Loading_Class

In [ ]:
# IMPORT CLASSES
sys.path.append(os.path.join(mainCodeDirectory,"3_Project_Algorithms","3_Tracked_Profiles"))
from CLASSES_TrackedProfiles import TrackedProfiles_DataLoading_CLASS, TrackedProfiles_Plotting_CLASS

In [ ]:
##############################################
#SETUP

In [ ]:
import sys
path=os.path.join(mainCodeDirectory,'Functions/')
sys.path.append(path)

import NumericalFunctions
from NumericalFunctions import * # import NumericalFunctions 
import PlottingFunctions
from PlottingFunctions import * # import PlottingFunctions

# # Get all functions in NumericalFunctions
# import inspect
# functions = [f[0] for f in inspect.getmembers(NumericalFunctions, inspect.isfunction)]
# functions

#####

#Import StatisticalFunctions 
import sys
dir2='/mnt/lustre/koa/koastore/torri_group/air_directory/Projects/DCI-Project/'
path=dir2+'Functions/'
sys.path.append(path)

import StatisticalFunctions
from StatisticalFunctions import * # import NumericalFunctions 

In [ ]:
##############################################
#DATA LOADING FUNCTIONS

In [ ]:
def MakeDataDictionary(variableNames,t,printstatement=False):
    timeString = ModelData.timeStrings[t]
    # print(f"Getting data from {timeString}","\n")
    
    dataDictionary = {variableName: CallLagrangianArray(ModelData, DataManager, timeString, variableName=variableName, printstatement=printstatement) 
                      for variableName in variableNames}      
    return dataDictionary
    
def GetSpatialData(t):    
    variableNames = ['Z']
    dataDictionary = MakeDataDictionary(variableNames,t)
    [Z] = (dataDictionary[k] for k in variableNames)
    return Z

In [ ]:
##############################################
#DATA LOADING

In [ ]:
#Loading in Tracked Parcels Info
trackedArrays,LevelsDictionary = TrackedParcel_Loading_Class.LoadingSubsetParcelData(ModelData,DataManager,
                                                         Results_InputOutput_Class)
LFC_profile = LevelsDictionary['LFC_profile'].copy()
hLines,hLineColors = TrackedProfiles_Plotting_CLASS.GetHLines(LevelsDictionary)

In [ ]:
##############################################
#PLOTTING FUNCTIONS FOR UPDRAFT AREA

In [ ]:
def ExtraAllAxisModifications(axes,variableNames):

    #matching UpdraftArea_g and UpdraftArea_c x-axises
    index1=variableNames.index('UpdraftArea_g')
    index2=variableNames.index('UpdraftArea_c')
    MatchAxisLimits([axes[index1], axes[index2]], dim='x')

    #snap axises to limits
    SnapLimitsToTicks(axes, dim='x')

    #setting left x limit
    for axis in axes:
        axis.set_xlim(left=0)

In [ ]:
# === Top Level: Make Figure and Plot all variables ===
def PlotAllVariables(profiles, profilesSE, variableNames, variableInfo,
                     parcelTypes, parcelDepths, ncols=2, figsize=(15, 5),
                     locationSubset=""):

    n_vars = len(variableNames)
    nrows = int(np.ceil(n_vars / ncols))

    fig = plt.figure(figsize=figsize)
    gs = gridspec.GridSpec(nrows, ncols, figure=fig, wspace=0.2, hspace=0.35)

    axes = [fig.add_subplot(gs[i // ncols, i % ncols]) for i in range(n_vars)]

    for i, var in enumerate(variableNames):
        axis = axes[i]
        TrackedProfiles_Plotting_CLASS.PlotSingleVariable(axis, profiles, profilesSE, var, variableInfo, parcelTypes, parcelDepths, hLines,hLineColors,
                                                          locationSubset=locationSubset)

    # Turn off unused axes (if any)
    for j in range(len(variableNames), nrows * ncols):
        fig.add_subplot(gs[j // ncols, j % ncols]).axis("off")

    TrackedProfiles_Plotting_CLASS.AddCategoryLegend(fig, parcelTypes)
    TrackedProfiles_Plotting_CLASS.AddDepthLegend(axes[0])

    ExtraAllAxisModifications(axes,variableNames)
    #extra: fixing top legend spacing
    fig.subplots_adjust(top=0.8)
    return fig

In [ ]:
def PlotAllPlots(trackedProfileArrays,trackedProfiles_SE,
                 timeIndiceRange,parcelTypes):

    variableNames = ["UpdraftArea_g", "UpdraftArea_c"]

    #Parcels Anywhere in Domain
    ################################ 
    fig = PlotAllVariables(
        profiles=trackedProfileArrays,
        profilesSE=trackedProfiles_SE,
        variableNames=variableNames,
        variableInfo=variableInfo,
        parcelTypes=parcelTypes,
        parcelDepths=["SHALLOW", "DEEP"],
        locationSubset=""
    )
    
    #saving
    folderName=f"Anywhere/{parcelTypes[0]}_vs_{parcelTypes[1]}"
    fileName=f"Tracked_Profiles_{dataName}{timeIndiceRange}" 
    SaveFigure(fig,plotType=f"Project_Algorithms/Tracked_Profiles/Tracked_Profiles_{dataName}",folderName=folderName,fileName=fileName)
    
    #Parcels Left of SBF
    ################################
    fig = PlotAllVariables(
        profiles=trackedProfileArrays,
        profilesSE=trackedProfiles_SE,
        variableNames=variableNames,
        variableInfo=variableInfo,
        parcelTypes=parcelTypes,
        parcelDepths=["SHALLOW", "DEEP"],
        locationSubset="_left"
    )
    
    #saving
    folderName=f"LEFT/{parcelTypes[0]}_vs_{parcelTypes[1]}"
    fileName=f"Tracked_Profiles_{dataName}{timeIndiceRange}_LEFT" 
    SaveFigure(fig,plotType=f"Project_Algorithms/Tracked_Profiles/Tracked_Profiles_{dataName}",folderName=folderName,fileName=fileName)
    
    #Parcels Right of SBF
    ################################
    fig = PlotAllVariables(
        profiles=trackedProfileArrays,
        profilesSE=trackedProfiles_SE,
        variableNames=variableNames,
        variableInfo=variableInfo,
        parcelTypes=parcelTypes,
        parcelDepths=["SHALLOW", "DEEP"],
        locationSubset="_right"
    )
    
    #saving
    folderName=f"RIGHT/{parcelTypes[0]}_vs_{parcelTypes[1]}"
    fileName=f"Tracked_Profiles_{dataName}{timeIndiceRange}_RIGHT" 
    SaveFigure(fig,plotType=f"Project_Algorithms/Tracked_Profiles/Tracked_Profiles_{dataName}",folderName=folderName,fileName=fileName)
    print("\n")

In [ ]:
##############################################
#PLOTTING SETUP FOR VARIABLES

In [ ]:
variableInfo = {
    "UpdraftArea_g": {
        "label": r"$A$",
        "units": r"($km^2$)",
        "multiplier": 1/(1e3**2)
    }, 
    "UpdraftArea_c": {
        "label": r"$A$",
        "units": r"($km^2$)",
        "multiplier": 1/(1e3**2)
    }
}

In [ ]:
##############################################
#PLOTTING FOR VARIABLES
timeIndiceRanges = ['',
                    '_10-12', '_12-14', '_14-17']
                    # '_10-11', '_11-12', '_12-13', '_13-14']
parcelTypesList = [["CL", "nonCL"],["SBF", "ColdPool"]]#,["turbulentCL", "Thermal"]]

In [ ]:
#Plotting
for parcelTypes in tqdm(parcelTypesList, desc="Parcel Groups"):
    for timeIndiceRange in tqdm(timeIndiceRanges, desc="Time Ranges", leave=False):
    
        #Updating MeanLFC
        LevelsDictionary = TrackedProfiles_Plotting_CLASS.UpdateMeanLFC(ModelData,timeIndiceRange,LevelsDictionary,LFC_profile)
        hLines,hLineColors = TrackedProfiles_Plotting_CLASS.GetHLines(LevelsDictionary)
        print(hLines)
        
        #Loading in Data
        dataName = "UpdraftArea"
        trackedProfileArrays = TrackedProfiles_DataLoading_CLASS.LoadProfile(ModelData,DataManager_TrackedProfiles, dataName, t=f'combined{timeIndiceRange}')
        trackedProfiles_SE = TrackedProfiles_DataLoading_CLASS.ExtractProfileStandardErrorArrays(trackedProfileArrays,ProfileStandardError)
        #Plotting
        PlotAllPlots(trackedProfileArrays,trackedProfiles_SE,
                    timeIndiceRange,parcelTypes)